In [1]:
import pandas as pd

In [5]:
import json

# Path to your JSON file
file_path = "/Users/sarvesh/Desktop/freelance/Stock-Bot/v2/financial_analysis_20250902_122854.json"

# Open and read the JSON file
with open(file_path, "r") as file:
    data = json.load(file)

# Print the data
print(data)



{'analysis_metadata': {'timestamp': '2025-09-02T12:28:54.288260', 'total_files_analyzed': 2, 'categories_analyzed': ['trading_data', 'video_content'], 'analysis_type': 'Financial Analyst'}, 'category_summaries': {'trading_data': {'summary': 'Analyzed 1 trading data files', 'key_findings': 'Financial metrics and performance indicators', 'files': ['apple_trading_data/spreadsheets/20250902_110205_Apple_Trading_Data_20250902_104928.xlsx']}, 'video_content': {'summary': 'Analyzed 1 video content files', 'key_findings': 'Visual and audio content analysis', 'files': ['apple_trading_data/videos/20250902_110206_appleq1.mp4']}}, 'overall_insights': {'total_files': 2, 'successful_analyses': 2, 'success_rate': '100.0%', 'analysis_coverage': 'Comprehensive financial analysis of Apple trading data across multiple formats', 'key_insights': 'Combined financial analysis of trading data, video content, and visual charts'}, 'detailed_results': {'apple_trading_data/spreadsheets/20250902_110205_Apple_Tradi

In [8]:
data['detailed_results']['apple_trading_data/spreadsheets/20250902_110205_Apple_Trading_Data_20250902_104928.xlsx']['analysis']

"Certainly! Let's dive into a comprehensive analysis of Apple (AAPL) based on the provided trading data.\n\n### 1. Price Action Analysis\n\n**Current Trend Direction and Strength:**\n- **Trend Direction:** The current trend for AAPL appears to be bullish. The stock has been making higher highs and higher lows over the past few months.\n- **Strength:** The strength of the trend can be considered strong, as evidenced by the consistent upward movement and positive momentum.\n\n**Key Support and Resistance Levels:**\n- **Support Levels:** \n  - Immediate Support: $165\n  - Secondary Support: $160\n- **Resistance Levels:**\n  - Immediate Resistance: $175\n  - Secondary Resistance: $180\n\n**Price Momentum Indicators:**\n- **Momentum:** The price momentum is positive, with the stock showing strong upward movement in recent weeks.\n\n### 2. Technical Indicators\n\n**Moving Averages (20, 50, 200 day):**\n- **20-Day Moving Average:** Currently at $168\n- **50-Day Moving Average:** Currently at 

In [9]:
data['detailed_results']['apple_trading_data/videos/20250902_110206_appleq1.mp4']['analysis']

'Financial analysis for apple_trading_data/videos/20250902_110206_appleq1.mp4: ### 1. Content Summary:\n- **Main Topics and Themes:**\n  - The video focuses on Apple\'s financial performance, specifically highlighting the company\'s earnings results and stock price trends.\n  - Key themes include Apple\'s quarterly earnings, its market position, and the impact on its stock price.\n  \n- **Key Messages Conveyed:**\n  - Apple reported strong financial results for its Q4 2023 earnings, surpassing expectations.\n  - The stock price of Apple (AAPL) has shown positive momentum following the earnings announcement.\n  - The company\'s performance is compared to the NASDAQ index and the S&P 500.\n\n- **Visual Elements and Presentation:**\n  - The video displays a stock chart for Apple (AAPL) with key annotations.\n  - There is a banner at the bottom left corner indicating "Apple Financial Results Q4 2023."\n  - The chart shows the stock price over time with green and red candlesticks representi

In [ ]:
import boto3
import os
import base64
import json
from typing import Optional
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

def process_video_with_nova_arn(
    video_path: str, 
    profile_arn: str, 
    prompt: str = "Analyze this video content",
    aws_profile: Optional[str] = None
) -> str:
    """
    Process video using AWS Bedrock Nova Pro with ARN profile
    
    Args:
        video_path: Path to the video file
        profile_arn: ARN of the model access policy
        prompt: Text prompt for video analysis
        aws_profile: Optional AWS profile name (instead of hardcoded keys)
    
    Returns:
        Analysis result or error message
    """
    
    try:
        # Initialize Bedrock client - use profile or credentials from .env
        if aws_profile:
            session = boto3.Session(profile_name=aws_profile)
            bedrock = session.client(
                service_name='bedrock-runtime',
                region_name='us-east-1'
            )
        else:
            # Use credentials from .env file
            bedrock = boto3.client(
                service_name='bedrock-runtime',
                region_name=os.getenv('AWS_DEFAULT_REGION', 'us-east-1'),
                aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
                aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
            )
        
        # Video file validation
        if not os.path.exists(video_path):
            return f"Error: Video file not found: {video_path}"
        
        file_size = os.path.getsize(video_path)
        video_name = os.path.basename(video_path)
        
        print(f"Processing video: {video_name}")
        print(f"File size: {file_size / (1024*1024):.2f} MB")
        print(f"Using profile ARN: {profile_arn}")
        
        # Check file size limit (25MB for Nova Pro)
        if file_size > 25 * 1024 * 1024:
            return f"Error: Video too large: {file_size / (1024*1024):.2f} MB (max 25MB)"
        
        # Read and encode video file
        print("Encoding video to base64...")
        with open(video_path, 'rb') as video_file:
            video_bytes = video_file.read()
            video_b64 = base64.b64encode(video_bytes).decode('utf-8')
        
        print(f"Video bytes length: {len(video_bytes)}")
        print(f"Base64 string length: {len(video_b64)}")
        print(f"Base64 starts with: {video_b64[:50]}...")
        
        # Determine video format from file extension
        file_ext = os.path.splitext(video_path)[1].lower()
        format_map = {
            '.mp4': 'mp4',
            '.mov': 'mov',
            '.avi': 'avi',
            '.webm': 'webm'
        }
        video_format = format_map.get(file_ext, 'mp4')
        
        print(f"Detected video format: {video_format}")
        print(f"File extension: {file_ext}")
        
        # Prepare the message content
        messages = [
        {
            "role": "user",
            "content": [
                {
                    "video": {
                        "format": video_format,  # Use the detected format
                        "source": {
                            "bytes": video_b64
                        }
                    }
                },
                {"text": prompt}
            ]
        }
    ]
        # Inference configuration
        inference_config = {
            "maxTokens": 1000,
            "temperature": 0.7,
            "topP": 0.9
        }
        
        print("Sending request to Bedrock Nova Pro...")
        
        # Try using invoke_model instead of converse for better video handling
        try:
            print("Trying invoke_model method...")
            response = bedrock.invoke_model(
                modelId="amazon.nova-pro-v1:0",
                body=json.dumps({
                    "messages": messages,
                    "inferenceConfig": inference_config
                }),
                contentType="application/json"
            )
            
            # Parse response
            response_body = json.loads(response['body'].read())
            result = response_body['output']['message']['content'][0]['text']
            
        except Exception as e:
            print(f"invoke_model failed: {str(e)}")
            print("Falling back to converse method...")
            
            # Fallback to converse method
            response = bedrock.converse(
                modelId="amazon.nova-pro-v1:0",
                messages=messages,
                inferenceConfig=inference_config
            )
            
            # Extract response text
            result = response['output']['message']['content'][0]['text']
        
        print("Analysis completed successfully")
        return result
        
    except FileNotFoundError:
        return f"Error: Video file not found: {video_path}"
    except Exception as e:
        return f"Error processing video: {str(e)}"

def validate_arn_format(arn: str) -> bool:
    """Validate ARN format"""
    return arn.startswith("arn:aws:bedrock:") and ("inference-profile" in arn or "model-access-policy" in arn)

# Usage example
if __name__ == "__main__":
    # Configuration
    profile_arn = "arn:aws:bedrock:ap-southeast-2:295386645352:inference-profile/apac.amazon.nova-pro-v1:0"
    video_file = "data/videos/appleq1.mp4"
    
    # Validate ARN format
    if not validate_arn_format(profile_arn):
        print("Invalid ARN format")
        exit(1)
    
    # Process video
    if os.path.exists(video_file):
        result = process_video_with_nova_arn(
            video_file, 
            profile_arn,
            "Analyze this video and provide key insights about the content, actions, and any notable elements."
        )
        print(f"\nAnalysis Result:")
        print("-" * 50)
        print(result)
    else:
        print(f"Video file not found: {video_file}")
        